# 10 — Encoder baselines: MizBERT and mBERT

The paper reports one recognizer, fine-tuned XLM-RoBERTa, with no comparison
point. This notebook adds two:

| Model | Why |
|---|---|
| `robzchhangte/MizBERT` | Mizo-specific BERT \citep{lalramhluna2024mizbert}, published in TALLIP. The natural competitor to a multilingual encoder, and the most conspicuous omission. |
| `bert-base-multilingual-cased` | The standard multilingual baseline XLM-R is usually compared against. |

Everything except the encoder is held constant: same `bio_v2` splits, same
learning rate, same effective batch, same epochs, same checkpoint-selection
rule. Sequence length is chosen per model by the same measurement rule, since
vocabularies differ.

**Run from the repository root.** Kernel: `Python (tka)`.
Budget two to three hours per model; each is saved as it finishes.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, sys, time, gc
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA   = ROOT / "data" / "processed" / "bio_v2"
MODELS = ROOT / "models"
RES    = ROOT / "results" / "ner"
MODELS.mkdir(exist_ok=True); RES.mkdir(parents=True, exist_ok=True)

BASELINES = {
    "mizbert": "robzchhangte/MizBERT",
    "mbert":   "bert-base-multilingual-cased",
}

for s in ("train", "dev", "test"):
    p = DATA / f"mizo_ner_{s}.json"
    print(("  ok   " if p.exists() else "  MISS ") + p.name)
    if not p.exists():
        sys.exit("Run 02b first")

ref = RES / "evaluation_v2.json"
print(("  ok   " if ref.exists() else "  MISS ") + "evaluation_v2.json (XLM-R reference)")
xlmr = json.load(open(ref, encoding="utf-8")) if ref.exists() else None

if torch.cuda.is_available():
    pr = torch.cuda.get_device_properties(0)
    VRAM = pr.total_memory / 1024**3
    print(f"\nGPU: {pr.name}  {VRAM:.1f} GB")
else:
    sys.exit("No GPU available")

def write_json(obj, path, **kw):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, **kw)

Repo root: C:\Users\Haulai\mizo-ner
  ok   mizo_ner_train.json
  ok   mizo_ner_dev.json
  ok   mizo_ner_test.json
  ok   evaluation_v2.json (XLM-R reference)

GPU: NVIDIA GeForce RTX 3060  12.0 GB


## Cell 2: Data and labels

In [2]:
def load_split(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

splits = {s: load_split(DATA / f"mizo_ner_{s}.json") for s in ("train", "dev", "test")}
for s, r in splits.items():
    print(f"  {s:<6}{len(r):>8,}")

entity_types = ["EVENT","FAC","GPE","LANGUAGE","LAW","LOC",
                "NORP","ORG","PERSON","PRODUCT","WORK_OF_ART"]
tag_list = ["O"] + [f"{p}-{e}" for e in entity_types for p in ("B","I")]
tag2id = {t: i for i, t in enumerate(tag_list)}
id2tag = {i: t for t, i in tag2id.items()}

seen = {t for r in splits.values() for x in r for t in x["tags"]}
assert not (seen - set(tag_list))
print(f"\n{len(tag_list)} labels, identical to the XLM-R run")

  train  352,941
  dev     44,118
  test    44,118

23 labels, identical to the XLM-R run


## Cell 3: Tokenizer inspection

Two things matter for a fair comparison. A model that lowercases its input
throws away capitalization, which is a strong entity cue, and that would be a
genuine disadvantage worth reporting rather than hiding. Vocabularies also
differ in how many subwords a Mizo word costs, so sequence length is measured
per model with the same rule used for XLM-R.

In [3]:
from transformers import AutoTokenizer

sample = [r["tokens"] for r in splits["train"][:20000]]
probe = "Aizawlah Liana leh MZU ṭhalai te an kal."

TOKS, MAXLEN = {}, {}
for key, name in BASELINES.items():
    try:
        tok = AutoTokenizer.from_pretrained(name)
    except Exception as e:
        print(f"  {key}: FAILED to load ({e})"); continue
    TOKS[key] = tok
    lower = getattr(tok, "do_lower_case", None)
    pieces = tok.tokenize(probe)
    L = np.array([len(tok(t, is_split_into_words=True)["input_ids"]) for t in sample])
    if (L > 96).mean() > 0.001:
        m = 128
    elif (L > 64).mean() > 0.001:
        m = 96
    else:
        m = 64
    MAXLEN[key] = m
    print(f"\n{key}  ({name})")
    print(f"  vocab {tok.vocab_size:,}   lowercases: {lower}")
    print(f"  subwords/sentence: mean {L.mean():.1f}  99th {np.percentile(L,99):.0f}  max {L.max()}")
    print(f"  MAX_LEN -> {m}   (XLM-R used 96)")
    print(f"  probe: {pieces[:14]}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]


mizbert  (robzchhangte/MizBERT)
  vocab 30,522   lowercases: True
  subwords/sentence: mean 15.8  99th 34  max 45
  MAX_LEN -> 64   (XLM-R used 96)
  probe: ['aizawlah', 'liana', 'leh', 'mzu', 'thalai', 'te', 'an', 'kal', '.']

mbert  (bert-base-multilingual-cased)
  vocab 119,547   lowercases: False
  subwords/sentence: mean 24.3  99th 57  max 83
  MAX_LEN -> 96   (XLM-R used 96)
  probe: ['Ai', '##za', '##wl', '##ah', 'Li', '##ana', 'le', '##h', 'M', '##Z', '##U', 'ṭ', '##hala', '##i']


## Cell 4: Dataset and metrics

In [4]:
from torch.utils.data import Dataset
from seqeval.metrics import f1_score, precision_score, recall_score

class NERDataset(Dataset):
    def __init__(self, recs, tok, max_len):
        self.recs, self.tok, self.max_len = recs, tok, max_len
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        rec = self.recs[i]
        enc = self.tok(rec["tokens"], is_split_into_words=True,
                       max_length=self.max_len, padding="max_length",
                       truncation=True, return_tensors="pt")
        wids, labels, prev = enc.word_ids(batch_index=0), [], None
        for w in wids:
            if w is None:            labels.append(-100)
            elif w != prev:          labels.append(tag2id[rec["tags"][w]])
            else:                    labels.append(-100)
            prev = w
        out = {"input_ids": enc["input_ids"].squeeze(0),
               "attention_mask": enc["attention_mask"].squeeze(0),
               "labels": torch.tensor(labels)}
        if "token_type_ids" in enc:
            out["token_type_ids"] = enc["token_type_ids"].squeeze(0)
        return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    T, P = [], []
    for pr_, lb in zip(preds, labels):
        t, p = [], []
        for a, b in zip(pr_, lb):
            if b != -100:
                t.append(id2tag[int(b)]); p.append(id2tag[int(a)])
        T.append(t); P.append(p)
    return {"precision": precision_score(T, P), "recall": recall_score(T, P),
            "f1": f1_score(T, P)}

BATCH, ACCUM = (64, 1) if VRAM >= 10 else (32, 2)
EPOCHS, LR = 5, 2e-5
print(f"batch {BATCH} x accum {ACCUM} = {BATCH*ACCUM}, lr {LR}, {EPOCHS} epochs")
print("identical to the XLM-R configuration")

batch 64 x accum 1 = 64, lr 2e-05, 5 epochs
identical to the XLM-R configuration


## Cell 5: Train and evaluate one encoder

In [6]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import classification_report

def run(key):
    name, tok, max_len = BASELINES[key], TOKS[key], MAXLEN[key]
    print(f"\n{'='*62}\n  {key}  ({name})  MAX_LEN={max_len}\n{'='*62}")
    model = AutoModelForTokenClassification.from_pretrained(
        name, num_labels=len(tag_list), id2label=id2tag, label2id=tag2id)
    print(f"parameters: {model.num_parameters()/1e6:.1f}M")

    ds = {s: NERDataset(splits[s], tok, max_len) for s in splits}
    args = TrainingArguments(
        output_dir=str(MODELS / f"_{key}_ckpt"),
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM, num_train_epochs=EPOCHS,
        weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        load_best_model_at_end=True, metric_for_best_model="f1",
        greater_is_better=True, save_total_limit=1, logging_steps=500,
        dataloader_num_workers=0, report_to="none", seed=42)

    trainer = Trainer(model=model, args=args,
                      train_dataset=ds["train"], eval_dataset=ds["dev"],
                      data_collator=DataCollatorForTokenClassification(tok),
                      compute_metrics=compute_metrics)
    t0 = time.time(); trainer.train(); hours = (time.time()-t0)/3600

    dest = MODELS / f"mizo_ner_{key}"
    trainer.save_model(str(dest)); tok.save_pretrained(str(dest))

    pred = trainer.predict(ds["test"])
    logits, labels = pred.predictions, pred.label_ids
    pi = np.argmax(logits, axis=2)
    T, P = [], []
    for pr_, lb in zip(pi, labels):
        t, p = [], []
        for a, b in zip(pr_, lb):
            if b != -100:
                t.append(id2tag[int(b)]); p.append(id2tag[int(a)])
        T.append(t); P.append(p)

    flat_t = [x for s in T for x in s]; flat_p = [x for s in P for x in s]
    acc_all = float(np.mean([a == b for a, b in zip(flat_t, flat_p)]))
    ent = [i for i, x in enumerate(flat_t) if x != "O"]
    acc_ent = float(np.mean([flat_t[i] == flat_p[i] for i in ent]))
    rep = classification_report(T, P, output_dict=True, digits=4)

    res = {
        "model": name, "max_len": max_len, "parameters_M": round(model.num_parameters()/1e6,1),
        "training_hours": round(hours,2),
        "token_accuracy_all": round(acc_all*100,2),
        "token_accuracy_entity": round(acc_ent*100,2),
        "f1_micro": round(float(f1_score(T,P)),4),
        "precision_micro": round(float(precision_score(T,P)),4),
        "recall_micro": round(float(recall_score(T,P)),4),
        "f1_macro": round(float(f1_score(T,P,average="macro")),4),
        "per_type": {k: {"precision": round(v["precision"],4),
                         "recall": round(v["recall"],4),
                         "f1": round(v["f1-score"],4),
                         "support": int(v["support"])}
                     for k, v in rep.items()
                     if k not in ("micro avg","macro avg","weighted avg")},
        "history": [{"epoch": round(h["epoch"]), "f1": h["eval_f1"]}
                    for h in trainer.state.log_history if "eval_f1" in h],
    }
    write_json(res, RES / f"baseline_{key}.json", indent=2)
    print(f"\n{key}: micro-F1 {res['f1_micro']:.4f}  macro-F1 {res['f1_macro']:.4f}  "
          f"({hours:.2f} h)")
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return res

print("run() defined")

run() defined


## Cell 6: MizBERT

In [7]:
res_mizbert = run("mizbert")


  mizbert  (robzchhangte/MizBERT)  MAX_LEN=64


config.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at robzchhangte/MizBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


parameters: 108.9M


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.093400,0.090493,0.817161,0.851769,0.834106
2,0.076500,0.076737,0.847951,0.865040,0.856410
3,0.063500,0.072399,0.857936,0.879129,0.868403
4,0.051200,0.071336,0.869489,0.881650,0.875528
5,0.044100,0.072747,0.869965,0.884904,0.877371



mizbert: micro-F1 0.8788  macro-F1 0.7274  (1.71 h)


## Cell 7: mBERT

In [8]:
res_mbert = run("mbert")


  mbert  (bert-base-multilingual-cased)  MAX_LEN=96


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


parameters: 177.3M


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.091000,0.090225,0.813355,0.855534,0.833912
2,0.075900,0.075388,0.850389,0.865688,0.857970
3,0.063700,0.070647,0.863195,0.879606,0.871323
4,0.051900,0.070318,0.871261,0.886727,0.878926
5,0.043500,0.071450,0.875865,0.890458,0.883101



mbert: micro-F1 0.8810  macro-F1 0.7347  (7.14 h)


## Cell 8: Comparison

In [9]:
rows = []
if xlmr:
    rows.append(("XLM-RoBERTa base", xlmr["overall"]["f1_micro"], xlmr["overall"]["f1_macro"],
                 xlmr["overall"]["token_accuracy_entity"], None))
for key, label in (("mizbert", "MizBERT"), ("mbert", "mBERT cased")):
    p = RES / f"baseline_{key}.json"
    if p.exists():
        r = json.load(open(p, encoding="utf-8"))
        rows.append((label, r["f1_micro"], r["f1_macro"],
                     r["token_accuracy_entity"], r["training_hours"]))

print(f"{'Encoder':<22}{'micro F1':>10}{'macro F1':>10}{'ent acc':>10}{'hours':>8}")
print("-" * 60)
for lab, mi, ma, ea, h in rows:
    hs = f"{h:.2f}" if h else "2.57"
    print(f"{lab:<22}{mi:>10.4f}{ma:>10.4f}{ea:>9.2f}%{hs:>8}")

print("\n% ---- Table: encoder comparison ----")
for lab, mi, ma, ea, h in rows:
    print(f"{lab:<22}& {ea:.2f}\\% & {mi:.4f} & {ma:.4f} \\\\")

Encoder                 micro F1  macro F1   ent acc   hours
------------------------------------------------------------
XLM-RoBERTa base          0.8739    0.7141    87.75%    2.57
MizBERT                   0.8788    0.7274    88.27%    1.71
mBERT cased               0.8810    0.7347    88.44%    7.14

% ---- Table: encoder comparison ----
XLM-RoBERTa base      & 87.75\% & 0.8739 & 0.7141 \\
MizBERT               & 88.27\% & 0.8788 & 0.7274 \\
mBERT cased           & 88.44\% & 0.8810 & 0.7347 \\


## Cell 9: Per-type comparison

In [10]:
if xlmr and (RES / "baseline_mizbert.json").exists():
    mz = json.load(open(RES / "baseline_mizbert.json", encoding="utf-8"))["per_type"]
    mb = (json.load(open(RES / "baseline_mbert.json", encoding="utf-8"))["per_type"]
          if (RES / "baseline_mbert.json").exists() else {})
    xl = xlmr["per_type"]
    order = sorted(xl, key=lambda k: -xl[k]["support"])
    print(f"{'Type':<14}{'support':>9}{'XLM-R':>9}{'MizBERT':>10}{'mBERT':>9}")
    print("-" * 51)
    for t in order:
        a = xl[t]["f1"]
        b = mz.get(t, {}).get("f1", float("nan"))
        c = mb.get(t, {}).get("f1", float("nan"))
        print(f"{t:<14}{xl[t]['support']:>9,}{a:>9.4f}{b:>10.4f}{c:>9.4f}")
else:
    print("run Cells 6 and 7 first")

Type            support    XLM-R   MizBERT    mBERT
---------------------------------------------------
PERSON           32,629   0.9169    0.9195   0.9213
GPE              11,495   0.8790    0.8870   0.8830
ORG              10,196   0.7917    0.7963   0.8092
NORP              1,942   0.7744    0.7976   0.7898
LOC                 700   0.6960    0.7099   0.7170
LANGUAGE            479   0.7688    0.8056   0.7850
WORK_OF_ART         381   0.8198    0.8090   0.8313
FAC                 327   0.5974    0.5724   0.6054
PRODUCT             291   0.4042    0.4305   0.4596
EVENT                90   0.5361    0.5775   0.5900
LAW                  68   0.6712    0.6963   0.6897
